# ragsynth v1 — toy-world iteration (SPEC §14)

**Question.** Does the packaged pipeline reproduce the prototype's 4-arm table —
i.e. do the fidelity / efficiency / validity meters separate the generation
strategies the way the design predicts?

**Setup.** `config.yaml` (frozen copy in this directory) = the shipped
`configs/v1_toy.yaml`: 8-component × 2-sub-mode world, d=64, 640 chunks,
5000 production queries (60/25/15 train/anchor/oracle), 500 records per arm,
`relabel_nearest` qrels (the §10 gate-style relabeling), n_boot=1000.

In [ ]:
import json
from pathlib import Path

HERE = Path.cwd() if Path("metrics.json").exists() else Path("experiments/v1")
arms = json.loads((HERE / "metrics.json").read_text())["arms"]

header = f"{'arm':<8}{'KL':>7}{'C2ST':>7}{'wC2ST':>7}{'MMD*1e3':>9}{'ESS/N':>7}" \
         f"{'tau':>7}{'tauAP':>7}{'p_drop':>8}{'p_noise':>8}"
print(header); print("-" * len(header))
for arm, b in arms.items():
    f, e, v = b["fidelity"], b["efficiency"], b["validity"]
    c = v["controls"]
    print(f"{arm:<8}{f['kl']:>7.3f}{f['c2st_auc']:>7.3f}{f['wc2st_mean']:>7.3f}"
          f"{f['mmd']*1e3:>9.2f}{e['ess_ratio']:>7.2f}{v['tau']:>7.3f}{v['tau_ap']:>7.3f}"
          f"{c['drop_index']['p_value']:>8.3f}{c['noise']['p_value']:>8.3f}")

## Reading the table

- **A2 is the only arm that matches within-cluster shape** (lowest wC2ST/MMD):
  quota matching (A1) fixes the cluster *marginals* (ESS ≈ oracle) but
  contributes nothing within-cluster (wC2ST ≈ 1) — exactly the
  post-stratification residual the theory predicts (SPEC §16).
- **A0 fails everything**: discriminable (C2ST ≈ 1), demand-mismatched
  (ESS ≈ 0.5), and the weakest system-ranking agreement.
- **A1 misses the injected noise regression** (p_noise > 0.05) while A2 and
  ORACLE detect it — a distributionally-off benchmark can be *insensitive to
  real regressions*; run positive controls (Sakai, SIGIR 2006).
- **ORACLE is the τ ceiling and it has a CI** — real-query subsamples are
  themselves noisy at this N.

In [ ]:
from IPython.display import Image, display
for fig in ("fidelity_bars", "ess_coverage", "tau_ci", "gate_rejects"):
    path = HERE / "figures" / f"{fig}.png"
    if path.exists():
        display(Image(filename=str(path)))

## Tuning trace (what it took to reproduce the prototype table)

1. **Cap every arm at n_per_arm=500** — uncapped arms kept all ~1300 gate
   survivors, tightening their bootstrap CIs vs the oracle and driving every
   control p-value to 0.
2. **A2 toy emission noise 0.15 → 0.42** — the exemplar-mean base loses the
   vMF(κ=400) dispersion that the prototype's z-sampling carries
   (≈0.39 perturbation norm); without it, dedup killed 60% of A2 and KL blew
   out to 2.4.
3. **`relabel_nearest` qrels** — the §10 gate-style nearest-chunk relabeling;
   restores the saturated-score dynamics behind both the sub-gate a0/a1 taus
   and the noise-control insensitivity story.
4. **p_group 0 for the toy quota arm** — chunk-group (multi-evidence) seeds
   average two chunks in the toy ChatModel; midpoint emissions have razor-thin
   nearest-gold margins, making 20% of A1's queries noise-fragile by
   construction (the prototype's A1 is single-chunk).

## Decision & next-iteration hypotheses (SPEC §16 triggers)

Open v2 items when observed on **real** data: R1 (scaled A2) if per-stratum
KL/C2ST stays high while quotas are matched; R2 (closed-loop prompt
optimization) when the gate pass-rate and τ plateau; R5 (graded qrels) if
τ_AP stalls below its gate while τ passes. The jsonl mini-corpus run
(`experiments/v1_local_corpus/`) is the template for the first real-corpus
iteration: swap the dataset block, keep the validator.